# Package

In [74]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [75]:
# ============================================================
# FIX PROJECT ROOT (CWD = d:\)
# ============================================================
from pathlib import Path
import pandas as pd
from utilsforecast.plotting import plot_series

PROJECT_ROOT = (
    Path.cwd()
    / "Portofolio Data science"
    / "Time Series"
    / "Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA"
)

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print("PROJECT_ROOT =", PROJECT_ROOT)

# ============================================================
# 0) Charger les 2 modèles (parquet) depuis PROJECT_ROOT
# ============================================================
path_ar1 = PROJECT_ROOT / "outputs" / "forecasts" / "unrate_ar_lag1_oos_forecasts.parquet"
path_arp = PROJECT_ROOT / "outputs" / "forecasts" / "unrate_ar_pstar_oos_forecasts.parquet"

df_ar1["date"] = pd.to_datetime(df_ar1["date"]).dt.tz_localize(None)
df_arp["date"] = pd.to_datetime(df_arp["date"]).dt.tz_localize(None)

PROJECT_ROOT = d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA


In [76]:
# ============================================================
# 1) Harmoniser clés (series_id/date) si besoin
# ============================================================
def ensure_keys(df):
    df = df.copy()
    if "unique_id" in df.columns and "series_id" not in df.columns:
        df = df.rename(columns={"unique_id": "series_id"})
    if "ds" in df.columns and "date" not in df.columns:
        df = df.rename(columns={"ds": "date"})
    return df

df_ar1 = ensure_keys(df_ar1)
df_arp = ensure_keys(df_arp)

# ============================================================
# 2) Sélection colonnes (cas standard: y_obs, y_hat_ar, y_hat_ar_lo_95, y_hat_ar_hi_95)
#    Si tes parquets ont exactement ces noms -> ça marche direct.
# ============================================================
need = ["series_id", "date", "y_obs", "y_hat_ar", "y_hat_ar_lo_95", "y_hat_ar_hi_95"]
missing_ar1 = [c for c in need if c not in df_ar1.columns]
missing_arp = [c for c in need if c not in df_arp.columns]
assert not missing_ar1, f"AR(1) parquet missing columns: {missing_ar1}"
assert not missing_arp, f"AR(p*) parquet missing columns: {missing_arp}"

base_ar1 = (
    df_ar1[need]
    .rename(columns={
        "y_hat_ar": "y_hat_ar1",
        "y_hat_ar_lo_95": "y_hat_ar1_lo_95",
        "y_hat_ar_hi_95": "y_hat_ar1_hi_95",
    })
)

base_arp = (
    df_arp[need]
    .rename(columns={
        "y_hat_ar": "y_hat_arp",
        "y_hat_ar_lo_95": "y_hat_arp_lo_95",
        "y_hat_ar_hi_95": "y_hat_arp_hi_95",
    })
    # on évite d'avoir 2 fois y_obs après merge
    .drop(columns=["y_obs"])
)

# ============================================================
# 3) Merge : une table unique avec AR1 + ARp
# ============================================================
df_ar_forecasts = base_ar1.merge(base_arp, on=["series_id", "date"], how="inner")

# ============================================================
# 4) Même STRUCTURE que ton code : df_obs + df_fcst
# ============================================================
df_obs = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",
        "y_obs": "y",
    })
    [["unique_id", "ds", "y"]]
)

df_fcst = (
    df_ar_forecasts
    .rename(columns={
        "series_id": "unique_id",
        "date": "ds",

        "y_hat_ar1": "AR1",
        "y_hat_ar1_lo_95": "AR1-lo-95",
        "y_hat_ar1_hi_95": "AR1-hi-95",

        "y_hat_arp": "ARp",
        "y_hat_arp_lo_95": "ARp-lo-95",
        "y_hat_arp_hi_95": "ARp-hi-95",
    })
    [[
        "unique_id", "ds",
        "AR1", "AR1-lo-95", "AR1-hi-95",
        "ARp", "ARp-lo-95", "ARp-hi-95",
    ]]
)

# ============================================================
# 5) Plot + rename légende (même logique que ton code)
# ============================================================
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

for trace in fig.data:
    n = (trace.name or "")
    nl = n.lower()

    if n == "y":
        trace.name = "Unemployment rate (%)"
    elif n == "AR1":
        trace.name = "AutoRegressive AR(1)"
    elif n == "ARp":
        trace.name = "AutoRegressive AR(p*)"
    elif "level_95" in nl:
        if "ar1" in nl:
            trace.name = "AR(1) 95% Prediction Interval"
        elif "arp" in nl:
            trace.name = "AR(p*) 95% Prediction Interval"
        else:
            trace.name = "95% Prediction Interval"

fig.show()

# Analyse des erreurs

In [77]:
import numpy as np
import pandas as pd
from math import sqrt, erf, isfinite
from typing import Iterable, Optional, Dict, List, Tuple

# ============================================================
# 1) DM test (sans SciPy) — HAC Bartlett
# ============================================================
def _phi(z: float) -> float:
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))

def dm_pvalue(loss_diff: np.ndarray, lags: int = 0) -> float:
    x = np.asarray(loss_diff, dtype=float)
    x = x[np.isfinite(x)]
    T = x.size
    if T < 3:
        return np.nan

    dbar = x.mean()
    gamma0 = np.dot(x - dbar, x - dbar) / T
    var = gamma0

    if lags > 0:
        for k in range(1, min(lags, T - 1) + 1):
            w = 1.0 - k / (lags + 1.0)
            cov = np.dot(x[k:] - dbar, x[:-k] - dbar) / T
            var += 2.0 * w * cov

    if var <= 0:
        return np.nan

    stat = dbar / sqrt(var / T)
    p = 2.0 * (1.0 - _phi(abs(stat)))
    return max(0.0, min(1.0, p))

In [78]:
# ============================================================
# 2) Pivot "MAE (p)" comme ton exemple
# ============================================================
def make_mae_dm_pivot(
    wide: pd.DataFrame,
    segments: List[Tuple[str, Optional[str], str]],
    *,
    methods: Optional[Iterable[str]] = None,  # None -> toutes sauf true
    include_overall: bool = True,
    overall_label: str = "Ensemble",
    min_obs: int = 20,
    round_digits: int = 4,
    add_dm: bool = True,
    dm_lags: int = 11,                        # ex h-1 si h=12
) -> pd.DataFrame:
    """
    wide: DataFrame index datetime OU colonne 'date', contient:
      - 'true' (observé)
      - colonnes modèles (AR1, ARP, LINREG, LightGBM, RIDGE, ...)
    segments: [(start, end_or_None, label), ...]
    """
    df = wide.copy()

    # index datetime
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
        df = df.set_index("date")
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("wide doit avoir un DatetimeIndex ou une colonne 'date'.")
    df = df.sort_index()

    if "true" not in df.columns:
        raise ValueError("wide doit contenir la colonne 'true'.")

    # méthodes
    if methods is None:
        meths = [c for c in df.columns if c != "true"]
    else:
        meths = [m for m in methods if m in df.columns and m != "true"]
    if len(meths) == 0:
        return pd.DataFrame()

    # fenêtres
    full_start, full_end = df.index.min(), df.index.max()
    windows: List[Tuple[pd.Timestamp, pd.Timestamp, str]] = []
    if include_overall:
        windows.append((full_start, full_end, overall_label))
    for start, end, label in segments:
        s = pd.to_datetime(start, utc=True)
        e = pd.to_datetime(end, utc=True) if end is not None else full_end
        windows.append((s, e, label))

    rows = []  # (model, period, cell)

    for start, end, label in windows:
        sub = df.loc[start:end, ["true"] + meths].copy().dropna(subset=["true"])

        # MAE par modèle + erreurs absolues
        maes: Dict[str, float] = {}
        err_abs: Dict[str, pd.Series] = {}
        for m in meths:
            diffs = (sub["true"] - sub[m]).abs().dropna()
            err_abs[m] = diffs
            maes[m] = float(diffs.mean()) if diffs.shape[0] >= min_obs else np.nan

        finite_models = [m for m in meths if isfinite(maes.get(m, np.nan))]
        best_m = min(finite_models, key=lambda k: maes[k]) if finite_models else None

        for m in meths:
            mae_val = maes.get(m, np.nan)
            if not isfinite(mae_val):
                rows.append((m, label, np.nan))
                continue

            # format "MAE (p)" si non-meilleur
            cell = f"{mae_val:.{round_digits}f}"
            if add_dm and best_m is not None and m != best_m:
                v1 = err_abs[m]
                v2 = err_abs[best_m]
                common = v1.index.intersection(v2.index)
                diff = (v1.loc[common] - v2.loc[common]).to_numpy()
                if diff.size >= min_obs:
                    pval = dm_pvalue(diff, lags=dm_lags)
                    if isfinite(pval):
                        cell = f"{cell} ({pval:.3f})"
            rows.append((m, label, cell))

    out = pd.DataFrame(rows, columns=["model", "period", "value"])
    pivot = out.pivot(index="model", columns="period", values="value")

    # ordre colonnes (comme ton output)
    desired_cols = ([overall_label] if include_overall else []) + [lbl for _, _, lbl in segments]
    pivot = pivot.reindex(columns=desired_cols)

    # ordre lignes = ordre methods
    pivot = pivot.reindex(index=meths)

    pivot.columns.name = "period"
    pivot.index.name = "model"
    return pivot

In [79]:
# ============================================================
# 3) Construire wide depuis df_ar_forecasts (AR1 + ARP)
#    (si tu veux ajouter LINREG/RIDGE/LightGBM, ajoute-les ici aussi)
# ============================================================
wide = (
    df_ar_forecasts
    .assign(date=pd.to_datetime(df_ar_forecasts["date"], utc=True, errors="coerce"))
    .rename(columns={
        "y_obs": "true",
        "y_hat_ar1": "AR1",
        "y_hat_arp": "ARP",
    })
    [["date", "true", "AR1", "ARP"]]
)

# ============================================================
# 4) Tes segments EXACTS
# ============================================================
segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2008-01-01", "2019-12-31", "2008-2019"),
    ("2019-01-01", None,         "2019-fin"),
]

methods_used = ["AR1", "ARP"]

table_pivot = make_mae_dm_pivot(
    wide=wide,
    segments=segments,
    methods=methods_used,
    include_overall=True,
    overall_label="Ensemble",
    min_obs=20,
    round_digits=4,
    add_dm=True,
    dm_lags=11,
)

print("Méthodes utilisées :", methods_used)
print(table_pivot)

Méthodes utilisées : ['AR1', 'ARP']
period        Ensemble       1990-1999       2000-2008       2008-2019  \
model                                                                    
AR1             0.6429          0.3118          0.3742  0.6382 (0.772)   
ARP     0.7038 (0.232)  0.4259 (0.091)  0.3836 (0.859)          0.6050   

period        2019-fin  
model                   
AR1             1.4162  
ARP     1.5884 (0.092)  


De 1990 à 2025, seule la période de 2008 à 2019 où ARp dégage une performance élevée que AR1. Voilà pourquoi, dans l'ensemble, ARp ne fait pas mieux que AR1. 

Cette performance n'est pas logique. En effet, un AR optimisé doit être plus performant que AR1. C'est ce qu'on a déjà vu dans les expériences passées. 